# Demographic, economic and crime correlations — Mexico City

**Author: Lorena Pérez · Phase 3 · TEAM_PLAN §4.4**

The primary analysis uses city-core urban AGEBs with fewer than 100 residents excluded. Pairwise-complete Pearson and Spearman tests are compared for three planned relationships. Distribution normality and Tukey-IQR outlier counts are checked for both variables in every pair. A second run includes low-population AGEBs to assess rate sensitivity. Results are descriptive associations, not causal effects.

All observations are loaded from the warehouse KPI view through `src.analysis.data.load_kpis`; the crime KPIs are based on the available January–July 2024 investigation-file snapshot.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/analysis/data.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the repository root containing src/analysis/data.py")
sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis.data import load_kpis

In [ ]:
pairs = [
    ("pop_density_km2", "business_density_km2", "Population density vs business density"),
    ("crime_rate_per_1k", "businesses_per_1k", "Crime rate vs businesses per 1,000 residents"),
    ("pea_rate", "crime_rate_per_1k", "Economically active population rate vs crime rate"),
]

primary = load_kpis(city_core_only=True, exclude_low_population=True)
including_low_population = load_kpis(city_core_only=True, exclude_low_population=False)
assert primary.crs.to_epsg() == 6372, f"Expected EPSG:6372, got {primary.crs}"
figures_dir = PROJECT_ROOT / "outputs/figures"
figures_dir.mkdir(parents=True, exist_ok=True)
print(f"Primary sample: {len(primary):,} AGEBs; sensitivity sample: {len(including_low_population):,} AGEBs")

In [ ]:
def iqr_outlier_count(values):
    q1, q3 = values.quantile([0.25, 0.75])
    spread = q3 - q1
    return int(((values < q1 - 1.5 * spread) | (values > q3 + 1.5 * spread)).sum())


def analyse_pairs(frame, sample_label):
    results = []
    for x_col, y_col, label in pairs:
        pair = frame[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
        if len(pair) < 8:
            raise ValueError(f"Only {len(pair)} complete observations for {label}; at least 8 are needed")
        x, y = pair[x_col], pair[y_col]
        x_normality = stats.normaltest(x).pvalue
        y_normality = stats.normaltest(y).pvalue
        pearson = stats.pearsonr(x, y)
        spearman = stats.spearmanr(x, y)
        results.append({
            "sample": sample_label,
            "relationship": label,
            "n": len(pair),
            "pearson_r": pearson.statistic,
            "pearson_p": pearson.pvalue,
            "spearman_rho": spearman.statistic,
            "spearman_p": spearman.pvalue,
            "x_normality_p": x_normality,
            "y_normality_p": y_normality,
            "x_iqr_outliers": iqr_outlier_count(x),
            "y_iqr_outliers": iqr_outlier_count(y),
        })
    return pd.DataFrame(results)


primary_results = analyse_pairs(primary, "low-population excluded")
sensitivity_results = analyse_pairs(including_low_population, "including low-population")
results = pd.concat([primary_results, sensitivity_results], ignore_index=True)
display(results.round(5))
print("Normality p-values use D’Agostino-Pearson; small values indicate departure from normality.")
print("Outliers are observations beyond 1.5 × IQR. Every coefficient and p-value uses its pairwise-complete n.")

In [ ]:
for x_col, y_col, label in pairs:
    pair = primary[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
    rho = stats.spearmanr(pair[x_col], pair[y_col]).statistic
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    ax.scatter(pair[x_col], pair[y_col], s=16, alpha=0.45, color="#2166ac", edgecolors="none")
    ax.set_xlabel(x_col.replace("_", " "))
    ax.set_ylabel(y_col.replace("_", " "))
    ax.set_title(f"{label}\nSpearman ρ={rho:.3f}; n={len(pair):,}")
    ax.grid(alpha=0.2)
    fig.tight_layout()
    path = figures_dir / f"31_correlation_{x_col}_vs_{y_col}.png"
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved {path}")

results_path = figures_dir / "31_correlation_results.csv"
results.to_csv(results_path, index=False)
print(f"Saved {results_path}")

## Interpretation

Use Spearman's rank coefficient as the primary description when normality tests reject normality, outliers are present, or the association is monotonic but not linear; Pearson's coefficient is included as a linear-association comparison. With thousands of AGEBs, p-values can be small for modest associations, so interpret coefficient magnitudes and compare the two population-threshold samples. Pairwise deletion means the reported sample size can differ between relationships. Correlation does not imply that business activity or demographic composition causes recorded crime.